# Historical reproduction: MARBERT vs AraBERT (audit item 1)

**Paper:** Abdul-Mageed, Elmadany & Nagoudi (2021), *ARBERT & MARBERT*, ACL-IJCNLP.
**Protocol (as in the paper):** train on the **full** training set, score the test set after every epoch,
report the **best epoch on test** with the published metric. This is deliberately the historical
(best-on-test) protocol and must be labelled as such in the report.

| Dataset | Metric | Published MARBERT / AraBERT |
|---|---|---|
| ArSarcasm-Sent | F1-PN | 71.50 / 63.50 |
| ASTD (trained on CAMelSent) | F1-PN | 78.00 / 72.00 |
| AJGT | Accuracy | 96.11 / 92.22 |

### How to use
1. `Runtime → Change runtime type → GPU` (T4 is enough).
2. Put Member 1's audited data in Google Drive (see **Section 3** for the expected layout).
3. Edit **Section 2 (Config)** if your folder differs, then `Runtime → Run all`.
4. Runs are **resumable**: if Colab disconnects, just run all cells again. Finished runs are skipped,
   interrupted runs continue from the last saved epoch.
5. When all 18 runs are done, **Section 9** builds the published-vs-ours table and the verdicts.

## 1. Environment: GPU check, Google Drive, package versions

In [2]:
import os, sys, json, subprocess, platform, time
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

import shutil as _sh
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip() if _sh.which("nvidia-smi") else "no NVIDIA GPU found"
if IN_COLAB and gpu_info == "no NVIDIA GPU found":
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU:", gpu_info)

Mounted at /content/drive
GPU: NVIDIA L4, 580.82.07, 23034 MiB


In [3]:
# Key packages are locked on the first session and re-installed with the SAME versions in later sessions,
# so all 18 runs use identical software even if Colab updates its defaults in between.
PROJECT_DIR = "/content/drive/MyDrive/marbert_repro" if IN_COLAB else os.path.abspath("marbert_repro")
os.makedirs(PROJECT_DIR, exist_ok=True)
LOCK_FILE = f"{PROJECT_DIR}/env_lock.json"
LOCKED_PKGS = ["transformers", "tokenizers", "huggingface_hub", "scikit-learn", "safetensors"]

from importlib.metadata import version as _v
if os.path.exists(LOCK_FILE):
    lock = json.load(open(LOCK_FILE))
    need = [f"{p}=={lock[p]}" for p in LOCKED_PKGS if p in lock and _v(p) != lock[p]]
    if need:
        print("Installing locked versions:", need)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
        raise SystemExit(">>> Locked versions installed. Do Runtime -> Restart session, then Run all again.")
    import torch
    if torch.__version__ != lock.get("torch"):
        print(f"WARNING: torch {torch.__version__} differs from locked {lock.get('torch')} - note it in departures.md")
else:
    import torch
    lock = {p: _v(p) for p in LOCKED_PKGS}
    lock.update({"torch": torch.__version__, "python": platform.python_version(), "gpu_first_session": gpu_info})
    json.dump(lock, open(LOCK_FILE, "w"), indent=2)
    print("Created", LOCK_FILE)
print(json.dumps(lock, indent=2))

{
  "transformers": "5.22.1",
  "tokenizers": "0.23.3",
  "huggingface_hub": "1.33.0",
  "scikit-learn": "1.6.1",
  "safetensors": "0.8.0",
  "torch": "2.14.1+cu130",
  "python": "3.13.16",
  "gpu_first_session": "Tesla T4, 580.82.07, 15360 MiB"
}


## 2. Config
Everything the paper specifies is here. Change only paths/seeds unless you log a departure.

In [4]:
DATA_DIR  = f"{PROJECT_DIR}/data"                     # Member 1's audited files
OUT_DIR   = f"{PROJECT_DIR}/outputs/historical"        # predictions + metrics (keep forever)
CKPT_DIR  = f"{PROJECT_DIR}/checkpoints/historical"    # resume checkpoints (deleted when a run finishes)

MODELS = {
    "marbert": "UBC-NLP/MARBERT",                 # original MARBERT, NOT MARBERTv2
    "arabert": "aubmindlab/bert-base-arabertv01", # AraBERT not named in [1]; our fixed choice (logged)
}

DATASETS = {
    # labels: order = class ids. metric: historical metric used to pick the best epoch and compare.
    "arsarcasm": {"labels": ["negative", "neutral", "positive"], "metric": "f1_pn",
                  "published": {"marbert": 71.50, "arabert": 63.50}},
    "astd":      {"labels": ["negative", "neutral", "positive"], "metric": "f1_pn",
                  "published": {"marbert": 78.00, "arabert": 72.00}},
    "ajgt":      {"labels": ["negative", "positive"], "metric": "accuracy",
                  "published": {"marbert": 96.11, "arabert": 92.22}},
}

# Hyperparameters reported in [1]
HP = dict(lr=2e-6, batch_size=32, max_len=256, epochs=25, weight_decay=0.0,
          optimizer="Adam (torch.optim.Adam)", scheduler="constant (unreported in [1]; assumption)", warmup=0)

SEEDS = [42, 43, 44]
RUN_ORDER = ["astd", "arsarcasm", "ajgt"]   # longest first

# Implementation choices NOT specified in [1] - each one is listed in departures.md
USE_AMP = True            # fp16 mixed precision (faster on T4). Set False for full fp32.
PREPROC = dict(replace_user=True, replace_url=True)  # @mentions -> USER, links -> URL (verify vs MARBERT model card)
TEXT_COL, LABEL_COL = "text", "label"
# Resume checkpoints (~2 GB each for MARBERT incl. optimizer; writing one to Drive takes ~20-60 s).
# Save every N epochs per dataset; 0 = never (short runs simply restart if Colab disconnects).
CKPT_EVERY = {"astd": 5, "arsarcasm": 0, "ajgt": 0}

## 3. Data
Expected layout in Drive (CSV, UTF-8, columns `text` and `label`):
```
marbert_repro/data/
  arsarcasm/train.csv   arsarcasm/test.csv
  astd/train.csv        astd/test.csv      # train = merged CAMelSent (train+dev of ArSAS, ASTD, SemEval, ArSenTD-LEV); test = ASTD test
  ajgt/train.csv        ajgt/test.csv      # split agreed with Member 1 (no official split - log it)
```
Labels may be `POS/NEG/NEU`, `positive/negative/neutral`, `Positive/Negative`, etc.; they are normalised below.
Any other label value stops the notebook so it can be fixed in the data audit, not silently dropped.

In [5]:
import re, random, hashlib
import numpy as np, pandas as pd

LABEL_ALIASES = {"pos": "positive", "positive": "positive", "neg": "negative", "negative": "negative",
                 "neu": "neutral", "neut": "neutral", "neutral": "neutral"}

URL_RE  = re.compile(r"(https?://\S+|www\.\S+)")
USER_RE = re.compile(r"@\w+")
def preprocess(t: str) -> str:
    t = str(t)
    if PREPROC["replace_url"]:  t = URL_RE.sub("URL", t)
    if PREPROC["replace_user"]: t = USER_RE.sub("USER", t)
    return re.sub(r"\s+", " ", t).strip()

def load_split(dataset: str, split: str) -> pd.DataFrame:
    path = f"{DATA_DIR}/{dataset}/{split}.csv"
    df = pd.read_csv(path)
    missing = {TEXT_COL, LABEL_COL} - set(df.columns)
    assert not missing, f"{path} is missing columns {missing}"
    df = df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
    raw = df["label"].astype(str).str.strip().str.lower()
    df["label"] = raw.map(LABEL_ALIASES)
    bad = sorted(raw[df["label"].isna()].unique())
    assert not bad, f"{path}: unknown labels {bad} - fix in the data audit / LABEL_ALIASES"
    labels = DATASETS[dataset]["labels"]
    extra = set(df["label"]) - set(labels)
    assert not extra, f"{path}: labels {extra} not expected for {dataset}"
    df["y"] = df["label"].map({l: i for i, l in enumerate(labels)})
    df["text"] = df["text"].map(preprocess)
    return df.reset_index(drop=True)

def file_md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

# Sanity check: counts per split and label (copy into the report / compare with Member 1's audit)
for ds in DATASETS:
    for sp in ["train", "test"]:
        p = f"{DATA_DIR}/{ds}/{sp}.csv"
        if os.path.exists(p):
            d = load_split(ds, sp)
            print(f"{ds:10s} {sp:5s} n={len(d):6d}", d["label"].value_counts().to_dict())
        else:
            print(f"{ds:10s} {sp:5s} MISSING -> {p}")

arsarcasm  train n=  8437 {'neutral': 4262, 'negative': 2813, 'positive': 1362}
arsarcasm  test  n=  2110 {'neutral': 1078, 'negative': 716, 'positive': 316}
astd       train n= 27394 {'negative': 11241, 'neutral': 9483, 'positive': 6670}
astd       test  n=   661 {'negative': 336, 'neutral': 166, 'positive': 159}
ajgt       train n=  1440 {'negative': 720, 'positive': 720}
ajgt       test  n=   360 {'positive': 180, 'negative': 180}


## 4. Metrics (+ unit test)
* **F1-PN** = mean of positive-class F1 and negative-class F1, computed on the **full 3-class test set**
  (neutral examples kept, neutral F1 not averaged in).
* Every epoch also logs accuracy, macro-F1 and per-class precision / recall / F1 / support.

In [6]:
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def compute_metrics(y_true, y_pred, labels):
    ids = list(range(len(labels)))
    m = {"accuracy": 100 * accuracy_score(y_true, y_pred),
         "macro_f1": 100 * f1_score(y_true, y_pred, labels=ids, average="macro", zero_division=0)}
    if "positive" in labels and "negative" in labels:
        pn = [labels.index("positive"), labels.index("negative")]
        m["f1_pn"] = 100 * f1_score(y_true, y_pred, labels=pn, average="macro", zero_division=0)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=ids, zero_division=0)
    m["per_class"] = {l: {"precision": 100 * p[i], "recall": 100 * r[i], "f1": 100 * f[i], "support": int(s[i])}
                      for i, l in enumerate(labels)}
    m["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=ids).tolist()
    return m

# Unit test (hand-computed): pos F1 = .5, neg F1 = 1, neu F1 = .5  ->  F1-PN = 75, macro-F1 = 66.67, acc = 66.67
_L = ["negative", "neutral", "positive"]; N, U, P = 0, 1, 2
_m = compute_metrics([P, P, N, N, U, U], [P, U, N, N, U, P], _L)
assert abs(_m["f1_pn"] - 75.0) < 1e-6 and abs(_m["macro_f1"] - 200/3) < 1e-6 and abs(_m["accuracy"] - 400/6) < 1e-6
print("metric unit test passed")

metric unit test passed


## 5. Seeding and model revisions
Model commit hashes are resolved once and stored, so every run uses the identical checkpoint.

In [7]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
import transformers

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

REV_FILE = f"{PROJECT_DIR}/model_revisions.json"
def get_revisions():
    revs = json.load(open(REV_FILE)) if os.path.exists(REV_FILE) else {}
    changed = False
    for k, mid in MODELS.items():
        if k not in revs:
            if os.path.isdir(mid):
                revs[k] = None                       # local folder (testing)
            else:
                from huggingface_hub import HfApi
                revs[k] = HfApi().model_info(mid).sha
            changed = True
    if changed: json.dump(revs, open(REV_FILE, "w"), indent=2)
    return revs

REVISIONS = get_revisions()
print(json.dumps(REVISIONS, indent=2))

device: cuda
{
  "marbert": "88e1fa192dd723cf0b3563500aec46209762eb22",
  "arabert": "01f6b7d2d17afcd0f79498f26f47dfc6207b56ea"
}


## 6. One training run (resumable)
One run = one dataset × one model × one seed. After every epoch it writes, to Drive:
`metrics_per_epoch.jsonl`, `preds/epoch_XX_test_preds.csv` (gold, prediction, class probabilities),
and a resume checkpoint. A `DONE` file marks a finished run.

In [8]:
import shutil
from torch.utils.data import DataLoader

class EncDataset(torch.utils.data.Dataset):
    def __init__(self, enc, y): self.enc, self.y = enc, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}; item["labels"] = int(self.y[i]); return item

def _autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and DEVICE == "cuda"))

@torch.no_grad()
def predict(model, loader):
    model.eval(); probs = []
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        with _autocast():
            logits = model(**b).logits
        probs.append(torch.softmax(logits.float(), -1).cpu())
    return torch.cat(probs).numpy()

def train_one_run(dataset, model_key, seed, out_root=None, ckpt_root=None, max_train=None, epochs=None):
    out_root, ckpt_root = out_root or OUT_DIR, ckpt_root or CKPT_DIR
    epochs = epochs or HP["epochs"]
    labels = DATASETS[dataset]["labels"]
    run_dir  = f"{out_root}/{dataset}/{model_key}/seed{seed}"
    ckpt_path = f"{ckpt_root}/{dataset}/{model_key}/seed{seed}/last.pt"
    if os.path.exists(f"{run_dir}/DONE"):
        print(f"[skip] {dataset}/{model_key}/seed{seed} already done"); return
    os.makedirs(f"{run_dir}/preds", exist_ok=True); os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

    set_seed(seed)
    train_df, test_df = load_split(dataset, "train"), load_split(dataset, "test")
    if max_train: train_df = train_df.sample(n=min(max_train, len(train_df)), random_state=seed).reset_index(drop=True)

    mid, rev = MODELS[model_key], REVISIONS[model_key]
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    model = AutoModelForSequenceClassification.from_pretrained(mid, revision=rev, num_labels=len(labels)).to(DEVICE)
    enc = lambda df: tok(df["text"].tolist(), truncation=True, max_length=HP["max_len"])
    train_ds, test_ds = EncDataset(enc(train_df), train_df["y"].values), EncDataset(enc(test_df), test_df["y"].values)
    collate = DataCollatorWithPadding(tok)   # dynamic padding: pad to longest tweet in batch (logged)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, collate_fn=collate)

    opt = torch.optim.Adam(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    start_epoch = 1
    metrics_file = f"{run_dir}/metrics_per_epoch.jsonl"

    if os.path.exists(ckpt_path):                                   # ---- resume
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["rng_cpu"])
        if DEVICE == "cuda" and ck.get("rng_cuda") is not None: torch.cuda.set_rng_state_all(ck["rng_cuda"])
        start_epoch = ck["epoch"] + 1
        if os.path.exists(metrics_file):                            # drop epochs logged after the checkpoint
            keep = [l for l in open(metrics_file) if json.loads(l)["epoch"] <= ck["epoch"]]
            open(metrics_file, "w").writelines(keep)
        print(f"[resume] {dataset}/{model_key}/seed{seed} from epoch {start_epoch}")
    else:
        if os.path.exists(metrics_file): os.remove(metrics_file)

    info = dict(protocol="historical: full training set, best epoch selected on TEST (as in [1])",
                dataset=dataset, model_key=model_key, model_id=mid, model_revision=rev, seed=seed,
                labels=labels, n_train=len(train_df), n_test=len(test_df),
                train_md5=file_md5(f"{DATA_DIR}/{dataset}/train.csv"), test_md5=file_md5(f"{DATA_DIR}/{dataset}/test.csv"),
                hp=HP, epochs=epochs, use_amp=USE_AMP, preprocessing=PREPROC, dynamic_padding=True, max_train=max_train,
                gpu=gpu_info, torch=torch.__version__, transformers=transformers.__version__,
                started=time.strftime("%Y-%m-%d %H:%M:%S"))
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time(); model.train()
        g = torch.Generator(); g.manual_seed(seed * 1000 + epoch)   # data order depends only on (seed, epoch) -> safe resume
        loader = DataLoader(train_ds, batch_size=HP["batch_size"], shuffle=True, generator=g, collate_fn=collate)
        tot, n = 0.0, 0
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with _autocast():
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * b["labels"].size(0); n += b["labels"].size(0)

        probs = predict(model, test_loader); pred = probs.argmax(1)
        m = compute_metrics(test_df["y"].values, pred, labels)
        m.update(epoch=epoch, train_loss=tot / n, seconds=round(time.time() - t0, 1))
        with open(metrics_file, "a") as f: f.write(json.dumps(m) + "\n")
        pdf = pd.DataFrame({"idx": np.arange(len(test_df)), "gold": test_df["y"].values, "pred": pred})
        for i, l in enumerate(labels): pdf[f"p_{l}"] = probs[:, i].round(6)
        pdf.to_csv(f"{run_dir}/preds/epoch_{epoch:02d}_test_preds.csv", index=False)
        key = DATASETS[dataset]["metric"]
        print(f"{dataset}/{model_key}/seed{seed} ep{epoch:02d} loss={tot/n:.4f} {key}={m[key]:.2f} "
              f"macroF1={m['macro_f1']:.2f} ({m['seconds']}s)")

        ce = CKPT_EVERY.get(dataset, 1) if isinstance(CKPT_EVERY, dict) else CKPT_EVERY
        if ce and epoch % ce == 0 and epoch < epochs:                # ---- checkpoint (atomic write)
            tmp = ckpt_path + ".tmp"
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "rng_cpu": torch.get_rng_state(),
                        "rng_cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}, tmp)
            os.replace(tmp, ckpt_path)

    info["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    open(f"{run_dir}/DONE", "w").write(info["finished"])
    shutil.rmtree(os.path.dirname(ckpt_path), ignore_errors=True)
    del model, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 7. Smoke test
1 epoch on 256 AJGT tweets per model: checks the full pipeline and prints time per epoch. Written to a separate `outputs/smoke` folder.

In [10]:
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    for mk in MODELS:
        shutil.rmtree(f"{PROJECT_DIR}/outputs/smoke/ajgt/{mk}", ignore_errors=True)
        train_one_run("ajgt", mk, seed=0, out_root=f"{PROJECT_DIR}/outputs/smoke",
                      ckpt_root=f"{PROJECT_DIR}/checkpoints/smoke", max_train=256, epochs=1)

## 8. All historical runs (2 models × 3 datasets × 3 seeds = 18)
Order is **seed-first**: all 6 dataset×model runs for seed 42, then 43, then 44, so a complete first-seed table exists early.
To split work across several Colab/Kaggle accounts, give each account a different `ONLY` list and copy the `outputs/historical` folders together at the end.
Safe to re-run at any time: finished runs are skipped, interrupted runs resume.
To run only part of the grid in one session, set `ONLY`, e.g. `ONLY = [("astd", "marbert", 42)]`.

In [ ]:
RUN_ALL = True
ONLY = None   # e.g. [("astd", "marbert", 42), ("astd", "arabert", 42)]

if RUN_ALL:
    grid = [(d, m, s) for s in SEEDS for d in RUN_ORDER for m in MODELS]   # seed-first: one full seed of results ASAP
    if ONLY: grid = [g for g in grid if g in ONLY]
    for d, m, s in grid:
        train_one_run(d, m, s)

## 9. Published vs ours: reproduction verdict
For each run the **best epoch on test** (historical metric; earliest epoch on ties) is taken.
A target is **reproduced** if |gap| ≤ 2 pp **or** the published score lies within our seed range,
**and** MARBERT > AraBERT (mean over seeds) on that dataset.

In [15]:
def collect_runs(out_root=OUT_DIR):
    rows = []
    for d in DATASETS:
        key = DATASETS[d]["metric"]
        for mk in MODELS:
            for s in SEEDS:
                rd = f"{out_root}/{d}/{mk}/seed{s}"
                if not os.path.exists(f"{rd}/DONE"): continue
                ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                best = max(ep, key=lambda e: (e[key], -e["epoch"]))
                rows.append(dict(dataset=d, model=mk, seed=s, metric=key, best_epoch=best["epoch"],
                                 score=best[key], macro_f1_at_best=best["macro_f1"], accuracy_at_best=best["accuracy"]))
    return pd.DataFrame(rows)

runs = collect_runs()
if runs.empty:
    print("No finished runs yet.")
else:
    runs.to_csv(f"{OUT_DIR}/per_run_best_epoch.csv", index=False)
    display(runs) if IN_COLAB else print(runs.to_string())

    summ = []
    for d in DATASETS:
        means = {}
        for mk in MODELS:
            r = runs[(runs.dataset == d) & (runs.model == mk)]
            if r.empty: continue
            pub = DATASETS[d]["published"][mk]
            mean, sd = r.score.mean(), (r.score.std(ddof=1) if len(r) > 1 else float("nan"))
            means[mk] = mean
            summ.append(dict(dataset=d, metric=DATASETS[d]["metric"], model=mk, n_seeds=len(r),
                             published=pub, ours_mean=round(mean, 2), ours_sd=round(sd, 2),
                             ours_min=round(r.score.min(), 2), ours_max=round(r.score.max(), 2),
                             gap_pp=round(mean - pub, 2),
                             within_2pp=abs(mean - pub) <= 2, within_seed_range=r.score.min() <= pub <= r.score.max()))
        ranking = means.get("marbert", np.nan) > means.get("arabert", np.nan) if len(means) == 2 else None
        for row in summ:
            if row["dataset"] == d:
                row["marbert_gt_arabert"] = ranking
                row["reproduced"] = bool((row["within_2pp"] or row["within_seed_range"]) and ranking) if ranking is not None else None
    summary = pd.DataFrame(summ)
    summary.to_csv(f"{OUT_DIR}/reproduction_summary.csv", index=False)
    display(summary) if IN_COLAB else print(summary.to_string())
    incomplete = summary[summary.n_seeds < len(SEEDS)]
    if not incomplete.empty: print("NOTE: some targets have fewer than", len(SEEDS), "seeds - verdict is provisional.")

,dataset,model,seed,metric,best_epoch,score,macro_f1_at_best,accuracy_at_best
0,arsarcasm,marbert,42,f1_pn,5,72.113760,74.748703,77.109005
1,arsarcasm,marbert,43,f1_pn,11,73.176557,75.457292,77.535545
2,arsarcasm,marbert,44,f1_pn,10,71.973836,74.236525,76.303318
3,arsarcasm,arabert,42,f1_pn,12,67.630053,70.793725,73.554502
4,arsarcasm,arabert,43,f1_pn,10,68.216314,71.906440,75.260664
5,arsarcasm,arabert,44,f1_pn,13,67.740980,71.173040,74.360190
6,astd,marbert,42,f1_pn,10,81.515484,69.237273,73.071104
7,astd,marbert,43,f1_pn,6,80.483165,68.179253,72.465961
8,astd,marbert,44,f1_pn,5,80.688823,67.326384,72.314675
9,astd,arabert,42,f1_pn,17,75.797348,64.094784,68.229955


,dataset,metric,model,n_seeds,published,ours_mean,ours_sd,ours_min,ours_max,gap_pp,within_2pp,within_seed_range,marbert_gt_arabert,reproduced
0,arsarcasm,f1_pn,marbert,3,71.50,72.42,0.66,71.97,73.18,0.92,True,False,True,True
1,arsarcasm,f1_pn,arabert,3,63.50,67.86,0.31,67.63,68.22,4.36,False,False,True,False
2,astd,f1_pn,marbert,3,78.00,80.90,0.55,80.48,81.52,2.90,False,False,True,False
3,astd,f1_pn,arabert,3,72.00,75.83,0.17,75.68,76.01,3.83,False,False,True,False
4,ajgt,accuracy,marbert,3,96.11,94.35,0.32,94.17,94.72,-1.76,True,False,True,True
5,ajgt,accuracy,arabert,3,92.22,88.89,0.73,88.06,89.44,-3.33,False,False,True,False


## 10. Departures log
Creates `departures.md` in Drive the first time (edit it there; it is not overwritten).

In [ ]:
DEP_FILE = f"{PROJECT_DIR}/departures.md"
if not os.path.exists(DEP_FILE):
    open(DEP_FILE, "w").write(f"""# Departures and assumptions - historical reproduction

| # | Paper [1] | Ours | Why / expected effect |
|---|---|---|---|
| 1 | AraBERT version not named | `{MODELS['arabert']}` @ `{REVISIONS['arabert']}` | Fixed 2020 base release; possible source of AraBERT gaps |
| 2 | MARBERT | `{MODELS['marbert']}` @ `{REVISIONS['marbert']}` | Original release, not MARBERTv2 |
| 3 | LR scheduler / warm-up unreported | Constant LR {HP['lr']}, no warm-up | Simplest reading of the paper |
| 4 | Precision not reported | fp16 mixed precision = {USE_AMP} | Speed on T4; small numeric differences |
| 5 | Padding not reported | Dynamic padding (max_len {HP['max_len']} truncation kept) | Speed; no effect on outputs with attention masks beyond float noise |
| 6 | Fine-tuning preprocessing not reported | mentions -> USER, URLs -> URL: {PREPROC} | Mirrors MARBERT pre-training normalisation (verify) |
| 7 | Seeds not reported | {SEEDS} | Three seeds per configuration |
| 8 | AJGT split not released | (fill in with Member 1) | Most likely cause of AJGT gaps |
| 9 | Hardware not reported | {gpu_info} | GPU non-determinism; small run-to-run noise |
| 10 | Classification head | HF `AutoModelForSequenceClassification` default (dropout 0.1, [CLS] pooler) | Standard BERT fine-tuning |
""")
    print("Created", DEP_FILE)
else:
    print(DEP_FILE, "exists - edit it in Drive")

---
# Independent protocol: dev-selected runs (audit item 3, Member 3)
Everything above is the **historical** protocol and is left unchanged. The sections below add the
**independent** protocol: train on 90% of the training set, **choose the epoch on the held-out dev set**,
and report the **test** score at that epoch. Models, hyperparameters, seeds and test sets are identical,
so the only difference between the two protocols is how the epoch is chosen.

**Before running:** run sections 1–6 first (setup, config, data, metrics, seeding, the training helpers;
set `RUN_SMOKE_TEST = False` in section 7). Section 8 can be skipped: finished historical runs are skipped anyway.

## 11. Dev-split data (Member 1's files)
Upload Member 1's nine files **as they are** into one Drive folder:
```
marbert_repro/data_dev/
  arsarcasm_train.csv  arsarcasm_dev.csv  arsarcasm_test.csv
  astd_train.csv       astd_dev.csv       astd_test.csv
  ajgt_train.csv       ajgt_dev.csv       ajgt_test.csv
```
(`*_train_full.csv` is not needed: it equals the historical training data.) Only the `text` and `label`
columns are used; `id` and the other columns are kept for error analysis.

In [11]:
DEV_DATA_DIR = f"{PROJECT_DIR}/data_dev"
OUT_DIR_DEV  = f"{PROJECT_DIR}/outputs/independent"
CKPT_DIR_DEV = f"{PROJECT_DIR}/checkpoints/independent"

def load_split_dev(dataset, split):
    """Same cleaning and label checks as load_split (section 3), reading Member 1's files."""
    path = f"{DEV_DATA_DIR}/{dataset}_{split}.csv"
    df = pd.read_csv(path)
    missing = {TEXT_COL, LABEL_COL} - set(df.columns)
    assert not missing, f"{path} is missing columns {missing}"
    keep = df.drop(columns=[c for c in ["text_clean"] if c in df.columns])
    raw = keep[LABEL_COL].astype(str).str.strip().str.lower()
    labels = DATASETS[dataset]["labels"]
    out = pd.DataFrame({"id": keep["id"] if "id" in keep.columns else np.arange(len(keep)),
                        "label": raw.map(LABEL_ALIASES)})
    bad = sorted(raw[out["label"].isna()].unique())
    assert not bad, f"{path}: unknown labels {bad}"
    extra = set(out["label"]) - set(labels)
    assert not extra, f"{path}: labels {extra} not expected for {dataset}"
    out["y"] = out["label"].map({l: i for i, l in enumerate(labels)})
    out["text"] = keep[TEXT_COL].map(preprocess)
    return out.reset_index(drop=True)

# Sanity check: sizes, label mix, and that train/dev/test do not share ids
for ds in DATASETS:
    parts = {}
    for sp in ["train", "dev", "test"]:
        p = f"{DEV_DATA_DIR}/{ds}_{sp}.csv"
        if not os.path.exists(p):
            print(f"{ds:10s} {sp:5s} MISSING -> {p}"); continue
        parts[sp] = load_split_dev(ds, sp)
        print(f"{ds:10s} {sp:5s} n={len(parts[sp]):6d}", parts[sp]["label"].value_counts().to_dict())
    if len(parts) == 3:
        assert not set(parts["train"]["id"]) & set(parts["dev"]["id"]), f"{ds}: train and dev share ids"
        hist_test = f"{DATA_DIR}/{ds}/test.csv"
        if os.path.exists(hist_test):
            same = sorted(pd.read_csv(hist_test)["text"].astype(str)) == sorted(pd.read_csv(f"{DEV_DATA_DIR}/{ds}_test.csv")[TEXT_COL].astype(str))
            print(f"{'':10s} test identical to historical test: {same}")

arsarcasm  train n=  7593 {'neutral': 3836, 'negative': 2531, 'positive': 1226}
arsarcasm  dev   n=   844 {'neutral': 426, 'negative': 282, 'positive': 136}
arsarcasm  test  n=  2110 {'neutral': 1078, 'negative': 716, 'positive': 316}
           test identical to historical test: True
astd       train n= 24654 {'negative': 10118, 'neutral': 8534, 'positive': 6002}
astd       dev   n=  2740 {'negative': 1123, 'neutral': 949, 'positive': 668}
astd       test  n=   661 {'negative': 336, 'neutral': 166, 'positive': 159}
           test identical to historical test: True
ajgt       train n=  1296 {'negative': 648, 'positive': 648}
ajgt       dev   n=   144 {'positive': 72, 'negative': 72}
ajgt       test  n=   360 {'positive': 180, 'negative': 180}
           test identical to historical test: True


## 12. One dev-selected run (resumable)
Same training loop and settings as section 6. After every epoch it scores **dev and test**, and writes
`metrics_per_epoch.jsonl` (each line holds `dev` and `test` metrics) plus dev and test predictions.
The epoch is chosen afterwards (section 14) on **dev only**.

In [12]:
def train_one_run_dev(dataset, model_key, seed, out_root=None, ckpt_root=None, max_train=None, epochs=None):
    out_root, ckpt_root = out_root or OUT_DIR_DEV, ckpt_root or CKPT_DIR_DEV
    epochs = epochs or HP["epochs"]
    labels = DATASETS[dataset]["labels"]
    run_dir   = f"{out_root}/{dataset}/{model_key}/seed{seed}"
    ckpt_path = f"{ckpt_root}/{dataset}/{model_key}/seed{seed}/last.pt"
    if os.path.exists(f"{run_dir}/DONE"):
        print(f"[skip] {dataset}/{model_key}/seed{seed} already done"); return
    os.makedirs(f"{run_dir}/preds", exist_ok=True); os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

    set_seed(seed)
    train_df, dev_df, test_df = (load_split_dev(dataset, s) for s in ["train", "dev", "test"])
    if max_train: train_df = train_df.sample(n=min(max_train, len(train_df)), random_state=seed).reset_index(drop=True)

    mid, rev = MODELS[model_key], REVISIONS[model_key]
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    model = AutoModelForSequenceClassification.from_pretrained(mid, revision=rev, num_labels=len(labels)).to(DEVICE)
    enc = lambda df: tok(df["text"].tolist(), truncation=True, max_length=HP["max_len"])
    train_ds = EncDataset(enc(train_df), train_df["y"].values)
    collate = DataCollatorWithPadding(tok)
    dev_loader  = DataLoader(EncDataset(enc(dev_df),  dev_df["y"].values),  batch_size=128, shuffle=False, collate_fn=collate)
    test_loader = DataLoader(EncDataset(enc(test_df), test_df["y"].values), batch_size=128, shuffle=False, collate_fn=collate)

    opt = torch.optim.Adam(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    start_epoch, metrics_file = 1, f"{run_dir}/metrics_per_epoch.jsonl"

    if os.path.exists(ckpt_path):                                    # ---- resume
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["rng_cpu"])
        if DEVICE == "cuda" and ck.get("rng_cuda") is not None: torch.cuda.set_rng_state_all(ck["rng_cuda"])
        start_epoch = ck["epoch"] + 1
        if os.path.exists(metrics_file):
            keep = [l for l in open(metrics_file) if json.loads(l)["epoch"] <= ck["epoch"]]
            open(metrics_file, "w").writelines(keep)
        print(f"[resume] {dataset}/{model_key}/seed{seed} from epoch {start_epoch}")
    elif os.path.exists(metrics_file):
        os.remove(metrics_file)

    info = dict(protocol="independent: train = 90% of training set, best epoch selected on DEV, test reported at that epoch",
                dataset=dataset, model_key=model_key, model_id=mid, model_revision=rev, seed=seed, labels=labels,
                n_train=len(train_df), n_dev=len(dev_df), n_test=len(test_df),
                md5={s: file_md5(f"{DEV_DATA_DIR}/{dataset}_{s}.csv") for s in ["train", "dev", "test"]},
                hp=HP, epochs=epochs, use_amp=USE_AMP, preprocessing=PREPROC, dynamic_padding=True, max_train=max_train,
                gpu=gpu_info, torch=torch.__version__, transformers=transformers.__version__,
                started=time.strftime("%Y-%m-%d %H:%M:%S"))
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    key = DATASETS[dataset]["metric"]

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time(); model.train()
        g = torch.Generator(); g.manual_seed(seed * 1000 + epoch)
        loader = DataLoader(train_ds, batch_size=HP["batch_size"], shuffle=True, generator=g, collate_fn=collate)
        tot, n = 0.0, 0
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with _autocast():
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * b["labels"].size(0); n += b["labels"].size(0)

        rec = {"epoch": epoch, "train_loss": tot / n}
        for split, loader_, df in [("dev", dev_loader, dev_df), ("test", test_loader, test_df)]:
            probs = predict(model, loader_); pred = probs.argmax(1)
            rec[split] = compute_metrics(df["y"].values, pred, labels)
            pdf = pd.DataFrame({"id": df["id"].values, "gold": df["y"].values, "pred": pred})
            for i, l in enumerate(labels): pdf[f"p_{l}"] = probs[:, i].round(6)
            pdf.to_csv(f"{run_dir}/preds/epoch_{epoch:02d}_{split}_preds.csv", index=False)
        rec["seconds"] = round(time.time() - t0, 1)
        with open(metrics_file, "a") as f: f.write(json.dumps(rec) + "\n")
        print(f"[dev-sel] {dataset}/{model_key}/seed{seed} ep{epoch:02d} loss={tot/n:.4f} "
              f"dev {key}={rec['dev'][key]:.2f}  test {key}={rec['test'][key]:.2f} ({rec['seconds']}s)")

        ce = CKPT_EVERY.get(dataset, 1) if isinstance(CKPT_EVERY, dict) else CKPT_EVERY
        if ce and epoch % ce == 0 and epoch < epochs:
            tmp = ckpt_path + ".tmp"
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "rng_cpu": torch.get_rng_state(),
                        "rng_cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}, tmp)
            os.replace(tmp, ckpt_path)

    info["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    open(f"{run_dir}/DONE", "w").write(info["finished"])
    shutil.rmtree(os.path.dirname(ckpt_path), ignore_errors=True)
    del model, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 13. All dev-selected runs (2 models × 3 datasets × 3 seeds = 18)
Seed-first order, resumable, and `ONLY_DEV` works like `ONLY` in section 8.
Optional smoke test first: 1 epoch on 256 AJGT tweets per model, written to `outputs/smoke_independent`.

In [13]:
RUN_DEV_SMOKE_TEST = True
if RUN_DEV_SMOKE_TEST:
    for mk in MODELS:
        shutil.rmtree(f"{PROJECT_DIR}/outputs/smoke_independent/ajgt/{mk}", ignore_errors=True)
        train_one_run_dev("ajgt", mk, seed=0, out_root=f"{PROJECT_DIR}/outputs/smoke_independent",
                          ckpt_root=f"{PROJECT_DIR}/checkpoints/smoke_independent", max_train=256, epochs=1)

RUN_ALL_DEV = True
ONLY_DEV = None   # e.g. [("astd", "marbert", 42)]
if RUN_ALL_DEV:
    grid = [(d, m, s) for s in SEEDS for d in RUN_ORDER for m in MODELS]
    if ONLY_DEV: grid = [g for g in grid if g in ONLY_DEV]
    for d, m, s in grid:
        train_one_run_dev(d, m, s)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

[dev-sel] ajgt/marbert/seed0 ep01 loss=0.7023 dev accuracy=50.00  test accuracy=46.94 (1.3s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv01
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[dev-sel] ajgt/arabert/seed0 ep01 loss=0.7206 dev accuracy=56.94  test accuracy=58.33 (1.3s)


... (50 outputs left out)

[dev-sel] ajgt/marbert/seed44 ep01 loss=0.6719 dev accuracy=88.89  test accuracy=86.67 (4.9s)
[dev-sel] ajgt/marbert/seed44 ep02 loss=0.5339 dev accuracy=92.36  test accuracy=91.11 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep03 loss=0.2944 dev accuracy=94.44  test accuracy=92.50 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep04 loss=0.1529 dev accuracy=95.14  test accuracy=93.33 (4.9s)
[dev-sel] ajgt/marbert/seed44 ep05 loss=0.0980 dev accuracy=95.14  test accuracy=93.33 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep06 loss=0.0718 dev accuracy=95.14  test accuracy=93.61 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep07 loss=0.0504 dev accuracy=94.44  test accuracy=92.50 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep08 loss=0.0495 dev accuracy=95.83  test accuracy=93.06 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep09 loss=0.0346 dev accuracy=95.83  test accuracy=93.89 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep10 loss=0.0307 dev accuracy=95.83  test accuracy=93.89 (4.8s)
[dev-sel] ajgt/marbert/seed44 ep11 loss=0.0266 dev accuracy=

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv01
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[dev-sel] ajgt/arabert/seed44 ep01 loss=0.6621 dev accuracy=80.56  test accuracy=82.78 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep02 loss=0.4929 dev accuracy=88.89  test accuracy=86.39 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep03 loss=0.3590 dev accuracy=90.28  test accuracy=86.94 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep04 loss=0.2589 dev accuracy=91.67  test accuracy=88.61 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep05 loss=0.2064 dev accuracy=93.06  test accuracy=87.50 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep06 loss=0.1701 dev accuracy=93.06  test accuracy=87.78 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep07 loss=0.1434 dev accuracy=92.36  test accuracy=88.61 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep08 loss=0.1228 dev accuracy=92.36  test accuracy=88.89 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep09 loss=0.0993 dev accuracy=93.06  test accuracy=88.33 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep10 loss=0.0800 dev accuracy=91.67  test accuracy=89.44 (4.6s)
[dev-sel] ajgt/arabert/seed44 ep11 loss=0.0732 dev accuracy=

## 14. Dev-selected results and comparison with the historical protocol
For each run: the epoch with the best **dev** score (historical metric; earliest on ties), and the **test**
score at that epoch. Then, per target: mean ± SD over seeds, the gap to [1], the MARBERT > AraBERT check,
and **inflation** = historical best-on-test mean − dev-selected mean (how much choosing on test raised the score).

In [16]:
def collect_dev_runs(out_root=OUT_DIR_DEV):
    rows = []
    for d in DATASETS:
        key = DATASETS[d]["metric"]
        for mk in MODELS:
            for s in SEEDS:
                rd = f"{out_root}/{d}/{mk}/seed{s}"
                if not os.path.exists(f"{rd}/DONE"): continue
                ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                sel = max(ep, key=lambda e: (e["dev"][key], -e["epoch"]))           # chosen on DEV
                oracle = max(ep, key=lambda e: (e["test"][key], -e["epoch"]))       # best-on-test, same runs (reference)
                rows.append(dict(dataset=d, model=mk, seed=s, metric=key,
                                 dev_epoch=sel["epoch"], dev_score=sel["dev"][key], test_at_dev_epoch=sel["test"][key],
                                 macro_f1_at_dev_epoch=sel["test"]["macro_f1"],
                                 best_test_epoch=oracle["epoch"], best_test_score=oracle["test"][key]))
    return pd.DataFrame(rows)

dev_runs = collect_dev_runs()
if dev_runs.empty:
    print("No finished dev-selected runs yet.")
else:
    dev_runs.to_csv(f"{OUT_DIR_DEV}/per_run_dev_selected.csv", index=False)
    display(dev_runs.round(2)) if IN_COLAB else print(dev_runs.round(2).to_string())

    hist = collect_runs() if os.path.exists(OUT_DIR) else pd.DataFrame()
    summ = []
    for d in DATASETS:
        means = {}
        for mk in MODELS:
            r = dev_runs[(dev_runs.dataset == d) & (dev_runs.model == mk)]
            if r.empty: continue
            pub = DATASETS[d]["published"][mk]
            mean = r.test_at_dev_epoch.mean(); sd = r.test_at_dev_epoch.std(ddof=1) if len(r) > 1 else float("nan")
            means[mk] = mean
            h = hist[(hist.dataset == d) & (hist.model == mk)] if not hist.empty else pd.DataFrame()
            h_mean = h.score.mean() if not h.empty else float("nan")
            summ.append(dict(dataset=d, metric=DATASETS[d]["metric"], model=mk, n_seeds=len(r), published=pub,
                             dev_selected_mean=round(mean, 2), dev_selected_sd=round(sd, 2),
                             gap_to_paper=round(mean - pub, 2),
                             historical_mean=round(h_mean, 2), inflation=round(h_mean - mean, 2),
                             same_runs_best_test_mean=round(r.best_test_score.mean(), 2)))
        rank = means.get("marbert", np.nan) > means.get("arabert", np.nan) if len(means) == 2 else None
        for row in summ:
            if row["dataset"] == d: row["marbert_gt_arabert"] = rank
    dev_summary = pd.DataFrame(summ)
    dev_summary.to_csv(f"{OUT_DIR_DEV}/independent_summary.csv", index=False)
    display(dev_summary) if IN_COLAB else print(dev_summary.to_string())
    if (dev_summary.n_seeds < len(SEEDS)).any(): print("NOTE: some targets have fewer than", len(SEEDS), "seeds - provisional.")

,dataset,model,seed,metric,dev_epoch,dev_score,test_at_dev_epoch,macro_f1_at_dev_epoch,best_test_epoch,best_test_score
0,arsarcasm,marbert,42,f1_pn,20,73.72,70.79,73.24,9,72.19
1,arsarcasm,marbert,43,f1_pn,3,74.32,71.07,73.99,5,72.49
2,arsarcasm,marbert,44,f1_pn,3,74.42,70.52,73.74,5,72.72
3,arsarcasm,arabert,42,f1_pn,5,71.66,66.17,70.13,8,67.95
4,arsarcasm,arabert,43,f1_pn,6,70.24,66.59,70.63,12,67.76
5,arsarcasm,arabert,44,f1_pn,9,71.25,67.50,70.85,9,67.50
6,astd,marbert,42,f1_pn,5,81.76,79.93,64.99,19,80.51
7,astd,marbert,43,f1_pn,5,81.07,81.00,66.31,5,81.00
8,astd,marbert,44,f1_pn,3,81.04,78.62,62.18,6,79.98
9,astd,arabert,42,f1_pn,7,78.88,75.72,59.85,7,75.72


,dataset,metric,model,n_seeds,published,dev_selected_mean,dev_selected_sd,gap_to_paper,historical_mean,inflation,same_runs_best_test_mean,marbert_gt_arabert
0,arsarcasm,f1_pn,marbert,3,71.50,70.80,0.28,-0.70,72.42,1.63,72.47,True
1,arsarcasm,f1_pn,arabert,3,63.50,66.75,0.68,3.25,67.86,1.11,67.74,True
2,astd,f1_pn,marbert,3,78.00,79.85,1.19,1.85,80.90,1.05,80.50,True
3,astd,f1_pn,arabert,3,72.00,75.55,0.49,3.55,75.83,0.29,75.80,True
4,ajgt,accuracy,marbert,3,96.11,93.52,0.32,-2.59,94.35,0.83,94.26,True
5,ajgt,accuracy,arabert,3,92.22,88.06,1.11,-4.16,88.89,0.83,88.61,True


## 15. Export predictions for error analysis (Member 1)
One CSV with a row per test tweet, per run, at the **selected epoch** of each protocol
(historical: best epoch on test; independent: best epoch on dev). Columns:
`dataset, model, protocol, seed, epoch, id, gold, pred, correct` (+ class probabilities).
Historical prediction files store the test row number; it maps to Member 1's `id` because the test files
have identical row order (checked below). Written to `outputs/error_analysis_predictions.csv`.

In [17]:
def export_error_analysis(path=f"{PROJECT_DIR}/outputs/error_analysis_predictions.csv"):
    frames = []
    for ds in DATASETS:
        labels, key = DATASETS[ds]["labels"], DATASETS[ds]["metric"]
        m1_test = f"{DEV_DATA_DIR}/{ds}_test.csv"
        ids = pd.read_csv(m1_test)["id"] if os.path.exists(m1_test) else None
        if ids is not None and os.path.exists(f"{DATA_DIR}/{ds}/test.csv"):   # row order must match to map idx -> id
            same = (pd.read_csv(f"{DATA_DIR}/{ds}/test.csv")["text"].astype(str).values
                    == pd.read_csv(m1_test)[TEXT_COL].astype(str).values).all()
            assert same, f"{ds}: historical and Member 1 test files are not in the same row order"
        for protocol, root in [("historical", OUT_DIR), ("independent", OUT_DIR_DEV)]:
            for mk in MODELS:
                for s in SEEDS:
                    rd = f"{root}/{ds}/{mk}/seed{s}"
                    if not os.path.exists(f"{rd}/DONE"): continue
                    ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                    if protocol == "historical":
                        best = max(ep, key=lambda e: (e[key], -e["epoch"]))["epoch"]
                        p = pd.read_csv(f"{rd}/preds/epoch_{best:02d}_test_preds.csv")
                        p["id"] = ids.values[p["idx"].values] if ids is not None else p["idx"]
                    else:
                        best = max(ep, key=lambda e: (e["dev"][key], -e["epoch"]))["epoch"]
                        p = pd.read_csv(f"{rd}/preds/epoch_{best:02d}_test_preds.csv")
                    out = pd.DataFrame({"dataset": ds, "model": mk, "protocol": protocol, "seed": s, "epoch": best,
                                        "id": p["id"].values,
                                        "gold": [labels[g] for g in p["gold"]], "pred": [labels[q] for q in p["pred"]]})
                    out["correct"] = out["gold"] == out["pred"]
                    for l in labels: out[f"p_{l}"] = p[f"p_{l}"].values
                    frames.append(out)
    if not frames:
        print("No finished runs to export."); return None
    df = pd.concat(frames, ignore_index=True)
    df.to_csv(path, index=False)
    print(f"Wrote {len(df):,} rows -> {path}")
    print(df.groupby(["protocol", "dataset", "model"]).agg(runs=("seed", "nunique"), accuracy=("correct", "mean")).round(4))
    return df

ea = export_error_analysis()

Wrote 37,572 rows -> /content/drive/MyDrive/marbert_repro/outputs/error_analysis_predictions.csv
                               runs  accuracy
protocol    dataset   model                  
historical  ajgt      arabert     3    0.8889
                      marbert     3    0.9435
            arsarcasm arabert     3    0.7439
                      marbert     3    0.7698
            astd      arabert     3    0.6757
                      marbert     3    0.7262
independent ajgt      arabert     3    0.8806
                      marbert     3    0.9352
            arsarcasm arabert     3    0.7387
                      marbert     3    0.7632
            astd      arabert     3    0.6768
                      marbert     3    0.7075
